In [65]:
#prepare ordinary kriging rasters as predictor variable for random forest models
#perform leave-one-out kriging to prevent target leakage for model
#remove stations with strong correlations (> 0.70) & within 1 pixel distance of coarsest predictor raster (~16 km)

# 1. Identify valid stations for day d and remove those within 1 pixel distance (~1.4 km) 
# 2. Calculate global daily residuals for all valid stations (residual = actual observation - seasonal linear trend).
# 3. Nscore transform these global residuals to obtain a Gaussian distribution.
# 4. Krige these residuals using seasonal variogram parameters to create a "Global Gaussian Residual Raster" (1 km).
# 5. **Loop within station i (LOO Point Prediction)**
    # Determine select stations for point i (exclude station i and any stations too close/correlated).
    # Read precipitation residuals of these select stations.
    # Nscore transform these residuals (building a lookup table specifically excluding station i).
    # Perform **Point Kriging** at the exact coordinate of station i to get a single Gaussian LOO value.
    # **Replace** the pixel value at station i's location in the "Global Gaussian Residual Raster" with this LOO value.
# **End station loop i**

# 6. Reverse nscore transform the entire modified "Global Gaussian Residual Raster".
# 7. Add the seasonal linear trend surface (1 km raster) back to the reversed residual raster.
# 8. Clamp all negative values in the final raster to 0.
# 9. Save the final kriged raster for day d as the predictor variable for Random Forest.


In [66]:
import os
import numpy as np
import pandas as pd
import geostatspy.geostats as geostats
import geostatspy.GSLIB as GSLIB
import matplotlib.pyplot as plt
import gc
from pyproj import Proj
from pyproj import Transformer
from pykrige.ok import OrdinaryKriging
import rasterio as rio
from rasterio.enums import Resampling
import xarray as xr
import rioxarray as rxr

In [67]:
# 1. Preprocessing 
# 1A. Get a pd dataframe that has station precipitation data along with coordinates (in both GCS and PCS)

temp_dir = r"..\Data\Processed\Kriging_Predictors\Temp"
template_wgs84 = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")
template_utm45 = rxr.open_rasterio(r"..\Data\Datasets\template_raster_UTM45N.tif")

# import all station observations and filter to include only training stations
all_prec = pd.read_csv(r'../Data/Processed/Insitu/insitu_rainfall_1991_2020.csv')
train_stations = pd.read_csv(r'../Data/Processed/Split_stations/train_stations.csv')
train_index = train_stations['index_no'].tolist()
stn_prec_df = all_prec[all_prec['index_no'].isin(train_index)].reset_index(drop=True)
stn_prec_df = stn_prec_df[stn_prec_df['year'] >= 2001]

#variogram parameters by season - range in km, a small nugget is kept in all seasons for kriging stability
# variogram_params = {
#     'winter': {'range': 17300, 'sill': 0.695, 'nugget': 0.005},
#     'pre_monsoon': {'range': 20100, 'sill': 1.083, 'nugget': 0.005},
#     'monsoon': {'range': 38100, 'sill': 0.845, 'nugget': 0.121},
#     'post_monsoon': {'range': 50400, 'sill': 1.184, 'nugget': 0.122}
# }
variogram_params = {
    'winter': {'range': 80300, 'sill': 0.695, 'nugget': 0.005},
    'pre_monsoon': {'range': 80000, 'sill': 1.083, 'nugget': 0.005},
    'monsoon': {'range': 80000, 'sill': 0.845, 'nugget': 0.121},
    'post_monsoon': {'range': 80000, 'sill': 1.184, 'nugget': 0.122}
}
seasons = {
    'winter': [12, 1, 2],
    'pre_monsoon': [3, 4, 5],
    'monsoon': [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}

stn_prec_df['season'] = (stn_prec_df['month'].apply(lambda x: [k for k, v in seasons.items() if x in v][0]))
stn_prec_df['date'] = pd.to_datetime(stn_prec_df[['year', 'month', 'day']])

def project_coordinates(df):
    myProj = Proj("+proj=utm +zone=45 +ellps=WGS84 +datum=WGS84 +units=m +no_defs")
    x, y = myProj(df['long'].values, df['lat'].values)
    df['X_m'] = x 
    df['Y_m'] = y 
    print("Coordinates projected to UTM Zone 45N (m)")
    return df

stn_prec_df = project_coordinates(stn_prec_df)
stn_prec_df = stn_prec_df[['date', 'index_no', 'lat', 'long', 'X_m', 'Y_m', 'season', 'rainfall_sum']]
print(stn_prec_df.head())


del all_prec, train_stations, train_index
gc.collect()

Coordinates projected to UTM Zone 45N (m)
           date  index_no       lat      long            X_m           Y_m  \
3653 2001-01-01       107  29.84496  80.54539 -124167.791697  3.319149e+06   
3654 2001-01-02       107  29.84496  80.54539 -124167.791697  3.319149e+06   
3655 2001-01-03       107  29.84496  80.54539 -124167.791697  3.319149e+06   
3656 2001-01-04       107  29.84496  80.54539 -124167.791697  3.319149e+06   
3657 2001-01-05       107  29.84496  80.54539 -124167.791697  3.319149e+06   

      season  rainfall_sum  
3653  winter           7.0  
3654  winter          26.0  
3655  winter           2.4  
3656  winter           0.0  
3657  winter           0.0  


187

In [68]:
template_utm45.y

<xarray.DataArray 'y' (y: 472)> Size: 4kB
array([3408444.628764, 3407356.252993, 3406267.877222, ..., 2897996.392164,
       2896908.016393, 2895819.640622], shape=(472,))
Coordinates:
  * y            (y) float64 4kB 3.408e+06 3.407e+06 ... 2.897e+06 2.896e+06
    spatial_ref  int64 8B 0

In [69]:
# 1B. add seasonal trend column to stn_prec_df and prepare global trend raster for each season for detrending and retrending

#spatial linear trend parameters by season - b1 = slope for lat, b2 = slope for long, c = intercept
trend_params = {
    'winter': {'b1': 25.7005, 'b2': 2.1057, 'c': -482.0265},
    'pre_monsoon': {'b1': 43.9990, 'b2': 35.0136, 'c': -3991.7142},
    'monsoon': {'b1': -79.2576, 'b2': 16.4877, 'c': 2026.6630},
    'post_monsoon': {'b1': -3.6445, 'b2': 3.7202, 'c': -156.1296}
}

unique_stations = stn_prec_df[['index_no', 'X_m', 'Y_m', 'lat', 'long']].drop_duplicates().reset_index(drop=True)
spatial_trend_df = unique_stations.copy()
for season, params in trend_params.items():
    b1 = params['b1']
    b2 = params['b2']
    c = params['c']
    spatial_trend_df.loc[:, f'trend_{season}'] = (b1 * spatial_trend_df['lat'] +
                                                   b2 * spatial_trend_df['long'] + c)


def create_trend_rasters_from_template(template_wgs84, trend_params, temp_dir):
    """Creates raster that capture spatial trend of precipitation for that season
        returned rasters are in WGS84
    """
    template_wgs84_s = template_wgs84.squeeze()
    os.makedirs(temp_dir, exist_ok=True)
    created_files = []
    for season, p in trend_params.items():
        trend_surface = (p['b1'] * template_wgs84_s.y) + (p['b2'] * template_wgs84_s.x) + p['c']
        
        trend_surface.name = f"trend_{season}"
        trend_surface.rio.write_crs(template_wgs84.rio.crs, inplace=True)
        trend_surface.rio.write_transform(template_wgs84.rio.transform(), inplace=True)
        
        out_name = os.path.join(temp_dir, f"global_trend_{season}.tif")
        trend_surface.rio.to_raster(out_name)
        created_files.append(out_name)
        print(f"Saved: {out_name}")

    return created_files

#create_trend_rasters_from_template(template_wgs84, trend_params, temp_dir)
print(spatial_trend_df.head())

del trend_params, b1, b2, c
gc.collect()

   index_no            X_m           Y_m       lat      long  trend_winter  \
0       107 -124167.791697  3.319149e+06  29.84496  80.54539    454.608322   
1       203  -85043.390932  3.252735e+06  29.26804  80.98360    440.703929   
2       215 -126834.271351  3.211280e+06  28.87623  80.57919    429.782649   
3       217  -59802.827982  3.236771e+06  29.13611  81.25009    437.874410   
4       306   30259.337403  3.278769e+06  29.55031  82.15428    450.423510   

   trend_pre_monsoon  trend_monsoon  trend_post_monsoon  
0         141.618262     989.231325           34.745403  
1         131.577669    1042.181695           38.478217  
2         100.178571    1066.567824           38.401682  
3         135.103655    1057.031957           39.950432  
4         184.986988    1039.111473           41.804648  


0

In [70]:
# Other Helper functions 

def perform_kriging(x_array, y_array, z_array, v_range, nugget, sill, chunk_size=500):
    grid_resolution = 1000 # 1 km
    x_min, x_max = template_utm45.x.min(), template_utm45.x.max()
    y_min, y_max = template_utm45.y.min(), template_utm45.y.max()
    
    grid_x = np.arange(x_min, x_max + grid_resolution, grid_resolution)
    grid_y = np.arange(y_min, y_max + grid_resolution, grid_resolution)
    
    OK = OrdinaryKriging(
        x_array, y_array, z_array,
        variogram_model='spherical',
        variogram_parameters={'range': v_range, 'sill': sill, 'nugget': nugget},
        enable_plotting=False,
        verbose=False,
        nlags = 20,
        coordinates_type='euclidean')
    
    # Execute kriging in chunks along both X and Y axes to avoid MemoryError
    z_interp_list = []
    
    for i in range(0, len(grid_y), chunk_size):
        chunk_y = grid_y[i:i + chunk_size]
        row_chunks = []
        for j in range(0, len(grid_x), chunk_size):
            chunk_x = grid_x[j:j + chunk_size]
            z_chunk, _ = OK.execute('grid', chunk_x, chunk_y)
            row_chunks.append(z_chunk)
        
        z_row = np.hstack(row_chunks)
        z_interp_list.append(z_row)
    
    z_interp = np.vstack(z_interp_list)
    return z_interp, grid_x, grid_y

def create_raster(value_array, grid_x, grid_y, save = False, output_path = temp_dir, filename = "null"):
    # Create DataArray with proper spatial coordinates
    z_da = xr.DataArray(
        value_array,
        coords={'y': grid_y, 'x': grid_x}, 
        dims=['y', 'x'], 
        name="interpolated_value"
    )
    z_da.rio.write_crs("epsg:32645", inplace=True)
    z_da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    # z_da = z_da.rio.reproject_match(template_utm45)
    
    if save == True:
        final_path = os.path.join(output_path, filename) 
        z_da.rio.to_raster(os.path.join(output_path, filename))
    
    return z_da

In [71]:
# first we find stations within 1 pixel distance

def calculate_euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

for i in range(len(unique_stations)):
    # Check if any other station is within 1 km
    current_station = unique_stations.iloc[i]
    for j in range(len(unique_stations)):
        if i != j:
            other_station = unique_stations.iloc[j]
            distance = calculate_euclidean_distance(
                current_station['X_m'], current_station['Y_m'],
                other_station['X_m'], other_station['Y_m']
            )
            if distance < 1410:
                print(f"Station {current_station['index_no']} is within 1.4 km of station {other_station['index_no']}")

#no stations found within 1 pixel 

del unique_stations, calculate_euclidean_distance, current_station, j, other_station, distance

In [72]:
#main function/loop

start_day = stn_prec_df['date'].min()
end_day = stn_prec_df['date'].max()
date_range_list = pd.date_range(start=start_day, end=end_day, freq='D').to_list()


# load seasonal trend rasters
trend_rasters = {
    s: rxr.open_rasterio(os.path.join(temp_dir, f"global_trend_{s}.tif")).squeeze()
    for s in seasons.keys()
}

for days in date_range_list[1:2]:
    prec_day = stn_prec_df[stn_prec_df['date'] == days].drop(columns='date').reset_index(drop=True)
    prec_day = prec_day.dropna(subset=['rainfall_sum'])
    if prec_day.empty:
        print(f"Skipping {days.date()}: No valid station data. Total skipped: {null_count}")
        continue
    
    season = prec_day['season'].iloc[0]
    
    merged_prec_trend_day = pd.merge(
        prec_day, spatial_trend_df, 
        on=['index_no', 'lat', 'long', 'X_m', 'Y_m'], 
        how='inner'
    )
    
    merged_prec_trend_day['residual'] = merged_prec_trend_day['rainfall_sum'] - merged_prec_trend_day[f'trend_{season}']
    merged_prec_trend_day = merged_prec_trend_day.dropna(subset=['residual']) 
    
    
    if len(merged_prec_trend_day) < 10:
        null_count = null_count + 1 
        print(f"null counter: {null_count}")
        continue
    
    z_nscore, nscore_vr, nscore_ns = geostats.nscore(pd.DataFrame({'val': merged_prec_trend_day['residual']}), 'val')   
    
    x = merged_prec_trend_day['X_m'].values
    y = merged_prec_trend_day['Y_m'].values
    
    # Perform Kriging to get 2D grid
    z_interp, grid_x, grid_y = perform_kriging(
        x, y, z_nscore, 
        variogram_params[season]['range'], 
        variogram_params[season]['nugget'], 
        variogram_params[season]['sill'])
    
    # [CORRECTION] Back transform z_interp (2D grid) to original scale
    
    # 1. Prepare Dataframe for backtr (it expects a DataFrame column)
    #    Flatten the 2D grid to 1D
    df_grid = pd.DataFrame({'nVal': z_interp.flatten()})
    
    # 2. Define tail parameters based on the day's residual range
    zmin = merged_prec_trend_day['residual'].min()
    zmax = merged_prec_trend_day['residual'].max()
    ltail = 1; ltpar = zmin  # Linear interpolation to lower limit
    utail = 1; utpar = zmax  # Linear interpolation to upper limit
    
    # 3. Perform back transformation
    #Pass nscore_vr (original) and nscore_ns (normal) as the transformation table
    z_transformed_flat = geostats.backtr(df_grid, 'nVal', nscore_vr, nscore_ns, zmin, zmax, ltail, ltpar, utail, utpar)
    
    # 4. Reshape back to 2D grid
    z_transformed = z_transformed_flat.reshape(z_interp.shape)
    
    # 
    residual_raster = create_raster(z_transformed, grid_x, grid_y, save=False, output_path=temp_dir, filename=f"residual_{days.date()}.tif")
    
    #reproject residual raster to WGS 84

    residual_raster = residual_raster.rio.reproject_match(template_wgs84)
    final_kriged_raster = trend_rasters[season] + residual_raster
    
    # Save with specific date
    final_kriged_raster.rio.to_raster(os.path.join(temp_dir, f"kriged_raster_{days.date()}.tif"))

